 # Trajectory vs Gamma Phase Analysis

## Sections

- [Helper Functions](#helper-functions): Functions needed throughtout the analysis, e.g. for loading data, fitting place fields, etc..
- [Python port](#python-port-for-pf-analysis) of the MATLAB script that recreates Figure 3 panels (B, C, G, H) from Pfeiffer & Foster, *Science*, 2015. Loops through 5 rats $\times$ 8 sessions (6 linear + 2 open field). Loads place fields, spike data, and trajectory event times. Plots are [here](#plotting).
- [Fire-rate Analysis](#analysis-of-fire-rates--spiking-behavior): Our parameters of our simulation look off compared to P&F data. Try to estimate important spiking characteristics:
    - [Peak firing rates of place fields](#place-fields-peak): Peak firing rates of place fields are important for the simulation, and we can estimate this from the data.
    - [Actual firing during events](#actual-firing-rates-during-trajectory-events-sanity-check): Just a sanity check. Maybe we can compare this to the firing rates in the simulations.
    - [Gamma modulation and min max spiking patterns](#gamma-phase-modulation-depth): Lorenz mentioned that this is overestimated in the simulations, so we should try to estimate this from the data.
    - [Place field fits](#place-field-width-with-gaussian-fits): Both for linear track and open field data from Pfeiffer & Foster, *Science*, 2015. 
- [Spearman Analysis](#spearman-rank-test): Done here in Python.
- [Movement Direction Analysis](#movement-direction-analysis)


In [ ]:
# imports and needed packages
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.io import loadmat
from scipy import stats
from scipy.optimize import curve_fit
from scipy.signal import butter, filtfilt
from pathlib import Path
import pandas as pd

# parameters and settings
# set this to the directory that contains Rat1/, Rat2/, ...
BASE_DIR = Path("data/")
# save figures? (set to a folder path to save, e.g. "figures/". To NONE to not save.)
SAVE_FIGS = "figs_pf_analysis"
if SAVE_FIGS is not None:
    SAVE_FIGS = Path(SAVE_FIGS)
    SAVE_FIGS.mkdir(exist_ok=True)
    
# save stats for place fields? I used parquet format, requires pyarrow or fastparquet package.
SAVE_STATS = False
# decode all sessions again if they not already decoded?
DECODE_ALL = True
# number of rats
RAT_RANGE = range(1, 6)
# in the linear track sessions there are some sessions where the track is oriented in the 
# opposite direction (y = short axis, x = long axis). Transpose the field data for those 
# sessions to make y the long axis?
DO_TRANSPOSE_LINEAR_TRACKS = True

LINEAR_FOLDERS = ["Linear1", "Linear2", "Linear3", "Linear4", "Linear5", "Linear6"]
OPEN_FOLDERS   = ["Open1",   "Open2"]
ALL_FOLDERS    = LINEAR_FOLDERS + OPEN_FOLDERS

DECODE_WINDOW = 0.020  # 20 ms window
DECODE_STEP   = 0.005  # 5 ms step

BIN_CM = 2.0  # spatial bin size of Field_Data (P&F use 2 cm, see _Data_Description.txt)
 
# spearman correlation panel settings
LM_BLUE      = "#3366FF"
MARKER_SIZE  = 6
MARKER_ALPHA = 0.4
FIT_LW       = 2


## helper functions

Just some functions that are used throughout the script.

In [ ]:
def circ_mean_deg(angles_deg):
    """
    Circular mean of angles given in degrees, returned in degrees.
    """
    a = np.asarray(angles_deg, dtype=float)
    a = a[~np.isnan(a)]
    if a.size == 0:
        return np.nan
    rad = np.deg2rad(a)
    return np.rad2deg(np.arctan2(np.mean(np.sin(rad)), np.mean(np.cos(rad))))


def load_session(session_path: Path):
    """
    Load the three .mat files for a session.

    Note: scipy.io.loadmat works for MATLAB files saved pre-v7.3. If a session was
    saved in v7.3 (HDF5), install `mat73` and replace these calls with mat73.loadmat.
    """
    field_mat = loadmat(session_path / "Field_Data.mat", squeeze_me=True)
    spike_mat = loadmat(session_path / "Spike_Data_With_Gamma_Phase.mat", squeeze_me=True)
    traj_mat  = loadmat(session_path / "Trajectory_Events.mat", squeeze_me=True)

    field_data         = np.asarray(field_mat["Field_Data"], dtype=float)
    spike_data         = np.asarray(spike_mat["Spike_Data"], dtype=float)
    excitatory_neurons = np.asarray(spike_mat["Excitatory_Neurons"]).ravel().astype(int)
    trajectory_events  = np.atleast_2d(np.asarray(traj_mat["Trajectory_Events"], dtype=float))
    return field_data, spike_data, excitatory_neurons, trajectory_events


def regularize_fields(field_data):
    """
    MATLAB row 34 onwards of Trajectory_Vs_Gamma_Phase_Analysis.m regularizes the field data by replacing non-positive values with a small positive value.
    We are doing the same here. Pretty much a one to one translation of the MATLAB code. 
    """
    field_data2 = field_data.copy()
    n_neurons = field_data2.shape[2]
    for n in range(n_neurons):
        f = field_data2[:, :, n]
        positive = f[f > 0]
        if positive.size == 0:
            # Neuron silent everywhere -> set to 1 so it drops out of the product.
            f[:] = 1.0
        else:
            m = positive.min()
            replacement = m / 10.0 if m / 10.0 > 0 else m
            f[f <= 0] = replacement
        field_data2[:, :, n] = f
    return field_data2


# decoding functions, adapted from MATLAB's Trajectory_Vs_Gamma_Phase_Analysis.m
def bayesian_decode(x, x_sum, decode_window=0.1):
    return (np.prod(x, axis=2)
                * np.exp(-decode_window * x_sum))


def decode_trajectory(field_data, field_sum, decode_window):
    decoded = bayesian_decode(field_data, field_sum, decode_window=decode_window)

    # Rare numerical-overflow rescue: divide all fields by 2^k until finite.
    if np.isinf(decoded).any():
        k = 1
        while np.isinf(decoded).any():
            scale = 2.0 ** k
            decoded = bayesian_decode(field_data / scale, field_sum / scale, decode_window=decode_window)
            k += 1

    decoded[decoded < 0] = 0.0
    total = decoded.sum()
    if total > 0:
        decoded = decoded / total
        # probability-weighted centroid. MATLAB used (col, row) ordering as
        # (x, y) with 1-based grid coords, so we mirror that here.
        ny, nx = decoded.shape
        x_grid, y_grid = np.meshgrid(np.arange(1, nx + 1),
                                     np.arange(1, ny + 1))
        max_x = np.sum(x_grid * decoded)
        max_y = np.sum(y_grid * decoded)
        peak = decoded.max()
    else:
        max_x = np.nan
        max_y = np.nan
        peak = np.nan
        
    return max_x, max_y, peak
    

# format for statiscal tests:
def _format_p(p, threshold=1e-10):
    return f"< {threshold:g}" if p < threshold else f"= {p:.3g}"


# helper functions for iterating sessions
def iter_sessions(subfolders):
    """Yield (rat, subfolder, dir) for every existing session directory."""
    for rat in RAT_RANGE:
        for sub in subfolders:
            d = BASE_DIR / f"Rat{rat}" / sub
            if d.is_dir():
                yield rat, sub, d




## Python Port for P&F Analysis

Decoding each trajectory in every condition in every rat:

In [ ]:

def decode_events(field_data, field_data2, spike_data, trajectory_events):
    """
    Bayesian decoding function of the MATLAB code in Trajectory_Vs_Gamma_Phase_Analysis.m, 
    adapted to Python and modified to also return 1D x/y step components and spike counts per step. 
    The main loop structure should be the same as in MATLAB. 
    """
    
    out = []
    windows = []
    
    # Field-data sums used in the Poisson exp() term (cached).
    field_sum = np.sum(field_data, axis=2)

    for ev in range(trajectory_events.shape[0]):
        t_start_event = trajectory_events[ev, 0]
        t_end         = trajectory_events[ev, 1]

        # mask spikes to those that occur during this event
        ev_mask   = (spike_data[:, 0] >= t_start_event) & (spike_data[:, 0] <= t_end)
        ev_spikes = spike_data[ev_mask]

        n_steps = int(np.ceil((t_end - t_start_event) / DECODE_STEP))
        seq = np.full((n_steps, 3), np.nan)  # columns: t, x, y

        line = 0
        t = t_start_event
        while t <= t_end:
            in_win = ev_spikes[(ev_spikes[:, 0] >= t) & (ev_spikes[:, 0] < t + DECODE_WINDOW)]
            if in_win.size > 0:
                # convert MATLAB 1 index to Python 0 index
                neuron_idx = in_win[:, 1].astype(int) - 1
                max_x, max_y, peak = decode_trajectory(field_data2[:, :, neuron_idx], field_sum, decode_window=DECODE_WINDOW)

                seq[line] = (t, max_x, max_y)
                line += 1
                windows.append((t, peak, in_win.shape[0], circ_mean_deg(in_win[:, 2]), ev))

            t += DECODE_STEP

        # pre loop check: if we have fewer than 2 steps, we can't compute step sizes or assign spikes to steps, so skip this event
        # MATLAB code does this post loop, via isempty()
        if line < 2:
            continue

        seq = seq[:line]

        # store step sizes per axis, 2D step size, spike counts, and gamma phases for each step
        # MATLAB code only stores 2D step size, spike counts, and gamma phases, but we can also store x/y components for our own analysis
        dt = np.diff(seq[:, 0])

        dx = np.diff(seq[:, 1])
        dy = np.diff(seq[:, 2])
        
        # dx = np.abs(dx)
        # dy = np.abs(dy)

        step_sizes_2d = np.sqrt(dx ** 2 + dy ** 2)

        spike_counts = np.zeros(line - 1)
        gamma_phases = np.full(line - 1, np.nan)
        for s in range(line - 1):
            t0 = seq[s, 0]
            t1 = seq[s + 1, 0] + DECODE_WINDOW
            step_spikes = ev_spikes[(ev_spikes[:, 0] >= t0) & (ev_spikes[:, 0] <= t1)]
            spike_counts[s] = step_spikes.shape[0]
            if step_spikes.shape[0] > 0:
                gamma_phases[s] = circ_mean_deg(step_spikes[:, 2])

        event_ids = np.full(line - 1, ev)
        out.append(np.column_stack([step_sizes_2d, dx, dy, spike_counts, gamma_phases, event_ids, dt]))

    steps = np.vstack(out) if out else np.empty((0, 7))
    win = np.asarray(windows, dtype=float) if windows else np.empty((0, 5))
    return steps, win

# run loop
OUTPUT_NAME = "All_Trajectory_Event_Step_Sizes_Gamma_Phases_And_Spike_Counts.npy"
WINDOW_NAME = "All_Trajectory_Window_PeakProb.npy"

for rat, sub, session_dir in iter_sessions(ALL_FOLDERS):
    out_path = session_dir / OUTPUT_NAME
    if out_path.exists() and not DECODE_ALL:
        print(f"Output file {out_path} already exists, skipping ...")
        continue

    print(f"Decoding {session_dir} ...", flush=True)

    field_data, spike_data, excitatory, traj_events = load_session(session_dir)

    # do sanity check for linear tracks. y = corridor length, x = corridor width. 
    if sub in LINEAR_FOLDERS:
        ny, nx, n_neurons = field_data.shape
        if ny < nx:
            print(f"Session {session_dir} has a linear track with y < x. Transpose field data to make y the long axis.")
            if DO_TRANSPOSE_LINEAR_TRACKS:
                field_data = field_data.transpose(1, 0, 2)

    # Restrict spikes to excitatory neurons (column 1 in Python = MATLAB col 2).
    spike_data = spike_data[np.isin(spike_data[:, 1].astype(int), excitatory)]
    field_data2 = regularize_fields(field_data)

    steps, win = decode_events(field_data, field_data2, spike_data, traj_events)
    np.save(out_path, steps)
    np.save(session_dir / WINDOW_NAME, win)


In [ ]:
COLUMNS = ["2Dsteps", "1Dsteps_x", "1Dsteps_y", "spikes", "phase", "event", "dt"]
WIN_COLUMNS = ["t", "peak_prob", "spikes", "phase", "event"]

def aggregate(subfolder_names, output_name=OUTPUT_NAME, columns=COLUMNS):
    chunks = []
    for rat in range(1, 6):
        for sub in subfolder_names:
            f = BASE_DIR / f"Rat{rat}" / sub / output_name
            if f.exists():
                df = pd.DataFrame(np.load(f), columns=columns)
                df["rat"] = rat
                df["session"] = sub
                df["event_uid"] = "Rat" + str(rat) + "_" + sub + "_" + df["event"].astype(int).astype(str)
                chunks.append(df)

    if not chunks:
        return pd.DataFrame(columns=columns + ["rat", "session", "event_uid"])
    data = pd.concat(chunks, ignore_index=True)
    # Wrap negative phases into [0, 360) for easy binning.
    data.loc[data["phase"] < 0, "phase"] += 360
    return data


linear_data    = aggregate(LINEAR_FOLDERS)
openfield_data = aggregate(OPEN_FOLDERS)

win_linear_data    = aggregate(LINEAR_FOLDERS, output_name=WINDOW_NAME, columns=WIN_COLUMNS)
win_openfield_data = aggregate(OPEN_FOLDERS, output_name=WINDOW_NAME, columns=WIN_COLUMNS)



In [ ]:

print(f"Linear rows:     {len(linear_data)}")
print(f"Open field rows: {len(openfield_data)}")

print("Linear data head:")
print(linear_data.head())

print("Open field data head:")
print(openfield_data.head())

# print unique events
print("Unique events in linear data:", linear_data["event_uid"].nunique())
print("Unique events in open field data:", openfield_data["event_uid"].nunique())

In [ ]:
print("Unique events in linear window data:", win_linear_data["event"].nunique())
print("Unique events in open field window data:", win_openfield_data["event"].nunique())

print("Linear window data head:")
print(win_linear_data.head())
print("Open field window data head:")
print(win_openfield_data.head())

### Plotting

For each phase bin (10° wide, 36 bins covering 0–360°) we compute the mean of the chosen column, then plot two cycles as bars overlaid with a smoothed line. MATLAB trick tiling the data?.

In [ ]:
def phase_binned_mean(data, value_col, n_bins=36, bin_width_deg=10, abs_value=False):
    bins = np.arange(0, n_bins * bin_width_deg + 1, bin_width_deg)  # bin edges
    labels = np.arange(n_bins)
    bin_idx = pd.cut(data["phase"], bins=bins, right=False, labels=labels)
    if abs_value:
        data = data.abs()
    return (
        data.groupby(bin_idx, observed=False)[value_col]
        .mean()
        .reindex(labels)          # preserve all bins, NaN where empty
        .to_numpy(dtype=float)
    )


def plot_phase_panel(
    data,
    value_col,
    n_bins=36,
    smooth_window=8,
    ylim=None,
    title=None,
    figsize=(6, 4),
    ax=None,
    ylabel=None
):
    binned = phase_binned_mean(data, value_col, n_bins=n_bins)

    tiled = np.tile(binned, 4)
    kernel = np.ones(smooth_window) / smooth_window
    smoothed = np.convolve(tiled, kernel, mode="same")
    smoothed = smoothed[n_bins:-n_bins]

    centers = np.arange(n_bins) * (360 / n_bins) + (360 / n_bins) / 2
    x2 = np.concatenate([centers, centers + 360])
    y2 = np.tile(binned, 2)

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    ax.bar(x2, y2, width=360 / n_bins, color="0.7", edgecolor="white")
    ax.plot(x2, smoothed, color="tab:red", lw=1.8)
    ax.set_xlim(10, 720)
    ax.set_xticks([10, 180, 360, 540, 720])
    ax.set_xlabel("Gamma phase [deg] for two cycles")
    ax.set_ylabel(ylabel if ylabel is not None else value_col)
    if ylim is not None:
        ax.set_ylim(ylim)
    if title is not None:
        ax.set_title(title)
    if standalone:
        fig.tight_layout()
    return fig, ax



# 2D step size vs. Phase

Linear track and Open Field. Images match P&F (2015) figures 3:


In [ ]:
plot_phase_panel(linear_data, value_col="2Dsteps", ylim=(2,  5), title="Linear Steps per Phase (3H)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_2dsteps_vs_phase_original.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_phase_panel(openfield_data, value_col="2Dsteps",  ylim=(3,  6.5),  title="Open Field Steps per Phase (3C)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_2dsteps_vs_phase_original.png", dpi=300, bbox_inches="tight")
plt.show()

# 1D step size vs. phase

x-step size (latitude) and y-step (longitude). Not included in P&F (2015):

In [ ]:
plot_phase_panel(linear_data, value_col="1Dsteps_x", title="Linear X Steps per Phase")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_1dsteps_x_vs_phase_based_on_pf.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_phase_panel(linear_data, value_col="1Dsteps_y", title="Linear Y Steps per Phase")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_1dsteps_y_vs_phase_based_on_pf.png", dpi=300, bbox_inches="tight")
plt.show()

## Spikes vs Phase

Images match figure 3 in P&F (2015):

In [ ]:
plot_phase_panel(linear_data, value_col="spikes", ylim=(15, 22), title="Linear Spikes per Phase (3G)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_spikes_vs_phase_original.png", dpi=300, bbox_inches="tight")
plt.show()




In [ ]:
plot_phase_panel(openfield_data, value_col="spikes", ylim=(14, 20), title="Open Field Spikes per Phase (3B)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_spikes_vs_phase_original.png", dpi=300, bbox_inches="tight")
plt.show()

## Analysis of fire rates / spiking behavior

### Place fields peak
Estimating `max_rate_Hz` for our simulation.
According to `_Data_Description.txt`, `Field_Data` is the smoothed firing rate in Hz, on 2-cm bins, with dimension order (Y, X, cell). Some cells are all-zero or all-NaN. These are skipped here:


In [ ]:
def collect_place_fields(subfolders):
    records = []
    for rat, sub, d in iter_sessions(subfolders):
        try:
            field_data, _, exc, _ = load_session(d)
            if DO_TRANSPOSE_LINEAR_TRACKS and sub in LINEAR_FOLDERS:
                ny, nx, _ = field_data.shape
                if ny < nx:
                    print(f"Session {d} has a linear track with y < x. Transpose field data to make y the long axis.")
                    field_data = field_data.transpose(1, 0, 2)
        except Exception as e:
            print(f"skipped {d}: {e}")  
            continue
        exc_idx = exc - 1  # MATLAB columns vs. Python
        exc_idx = exc_idx[(exc_idx >= 0) & (exc_idx < field_data.shape[2])]
        for n in exc_idx:
            f = field_data[:, :, n]
            if not np.any(np.isfinite(f) & (f > 0)):
                continue
            records.append(dict(
                rat=rat,
                session=sub, 
                neuron=int(n),
                field=f, 
                peak_Hz=float(np.nanmax(f)),
                mean_Hz=float(np.nanmean(f[f > 0])),
            ))
    return records

linear_fields = collect_place_fields(LINEAR_FOLDERS)
open_fields   = collect_place_fields(OPEN_FOLDERS)
print(f"Linear: {len(linear_fields)} place fields")
print(f"Open: {len(open_fields)}  place fields")

linear_peaks = np.array([r["peak_Hz"] for r in linear_fields])
print(pd.DataFrame({"Linear peaks (Hz)": linear_peaks}).describe())

open_peaks   = np.array([r["peak_Hz"] for r in open_fields])
print(pd.DataFrame({"Open peaks (Hz)": open_peaks}).describe())

# save place field stats to parquet
if SAVE_STATS:
    pd.DataFrame(linear_fields).to_parquet("linear_place_fields.parquet", index=False)
    pd.DataFrame(open_fields).to_parquet("openfield_place_fields.parquet", index=False)

### Distribution of peak firing rates across cells

Plotted on linear and log-y axes. The log-panel makes it obvious whether the tail is exponential...

In [ ]:
def plot_rate_distribution(peaks, title, fit_exponential=False):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    if fit_exponential:
        x = np.linspace(0, peaks.max(), 100)
    
    for ax, log in zip(axes, [False, True]):
        ax.hist(peaks, bins=60, density=True, color="0.7", edgecolor="white")
        if fit_exponential:
            # Fit exponential distribution to the data
            loc, scale = stats.expon.fit(peaks, floc=0)  # force location to 0
            pdf = stats.expon.pdf(x, loc=loc, scale=scale)
            ax.plot(x, pdf, color="r", linewidth=2, label=f"Exp fit ($\\lambda$={1/scale:.2f})")
        if log: 
            ax.set_yscale("log")
        ax.set_xlabel("Peak firing rate (Hz)")
        ax.set_ylabel("density" + (" (log)" if log else ""))

        # Add vertical lines for mean and median
        ax.axvline(np.median(peaks), color="r", lw=2,
                   label=f"median={np.median(peaks):.2f}")
        ax.axvline(np.mean(peaks),   color="b", lw=2, ls="--",
                   label=f"mean={np.mean(peaks):.2f}")
        ax.legend(loc="upper right") 
        
    if title is not None:
        fig.suptitle(title)
    
    fig.tight_layout()
    return fig

plot_rate_distribution(linear_peaks, "Linear track | Peak firing rate per cell", fit_exponential=False)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_peak_rate_distribution.png", dpi=300, bbox_inches="tight")

plot_rate_distribution(open_peaks, "Open field | Peak firing rate per cell", fit_exponential=False)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_peak_rate_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# log normal alternative - fit log-normal distribution to the data
def plot_log_normal_fit(peaks, title, x_label):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(peaks, bins=60, density=True, color="0.7", edgecolor="white")
    
    # Fit log-normal distribution to the data
    shape, loc, scale = stats.lognorm.fit(peaks, floc=0)
    x = np.linspace(peaks.min(), peaks.max(), 500)
    pdf = stats.lognorm.pdf(x, shape, loc=loc, scale=scale)
    ax.plot(x, pdf, color="r", linewidth=2,
            label=f"LogNorm fit ($\\sigma={shape:.2f}$, $\\mu={np.log(scale):.2f}$, $median={scale:.2f}$)")
    
    ax.set_xlabel(x_label)
    ax.set_ylabel("density")
    ax.legend(loc="upper right") 
    if title is not None:
        ax.set_title(title)
    
    fig.tight_layout()
    return fig

plot_log_normal_fit(linear_peaks, "Linear track | Peak firing rate per cell", x_label="Peak firing rate (Hz)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_peak_rate_distribution_log_normal_fit.png", dpi=300, bbox_inches="tight")


In [ ]:
plot_log_normal_fit(open_peaks, "Open field | Peak firing rate per cell", x_label="Peak firing rate (Hz)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_peak_rate_distribution_log_normal_fit.png", dpi=300, bbox_inches="tight")
plt.show()

### Actual firing rates during trajectory events (sanity check)

Counts spikes during the trajectory events:

* `replay_mean_Hz`: total spikes / total event duration, across the whole session. 
* `participation_rate`: fraction of events in which the cell fires at least one spike. Reflects how sparse replay is per cell. 
* `spikes_per_event_active`: mean spike count across **participating** events only. Sanity check here for max_rate_Hz in our simulation. If replay_mean_Hz is much lower than max_rate_Hz, then we should expect a more than one spikes_per_event_rate.

These three are useful as sanity checks on the simulated output: if `max_rate_Hz` and `sigma_code_cm` are fitted into the simulation, the simulated events should reproduce roughly the same participation rate and spikes-per-active-event you see here, right?


In [ ]:
def replay_rates(subfolders):
    rows = []
    for rat, sub, d in iter_sessions(subfolders):
        try:
            _, spike_data, exc, traj = load_session(d)
        except Exception as e:
            print(f"  skip {d}: {e}");  continue
        spk = spike_data[np.isin(spike_data[:, 1].astype(int), exc)]
        total_event_sec = float(np.sum(traj[:, 1] - traj[:, 0]))
        n_events        = traj.shape[0]
        if total_event_sec <= 0 or n_events == 0:
            continue

        for n in exc:
            spk_n  = spk[spk[:, 1].astype(int) == n, 0]
            counts = np.fromiter(
                (np.sum((spk_n >= t0) & (spk_n <= t1)) for t0, t1 in traj[:, :2]),
                dtype=int, count=n_events,
            )
            active = counts > 0
            rows.append(dict(
                rat=rat, session=sub, neuron=int(n),
                replay_mean_Hz         = counts.sum() / total_event_sec,
                participation_rate     = active.mean(),
                spikes_per_event_active= counts[active].mean() if active.any() else np.nan,
            ))
    return pd.DataFrame(rows)

linear_replay = replay_rates(LINEAR_FOLDERS)
open_replay   = replay_rates(OPEN_FOLDERS)

print(pd.DataFrame({
    "Linear mean (Hz)":      linear_replay["replay_mean_Hz"],
    "Linear participation":  linear_replay["participation_rate"],
    "Linear spk/active ev":  linear_replay["spikes_per_event_active"],
    "Open mean (Hz)":        open_replay["replay_mean_Hz"],
    "Open participation":    open_replay["participation_rate"],
    "Open spk/active ev":    open_replay["spikes_per_event_active"],
}).describe())


In [ ]:
metrics = [
    ("replay_mean_Hz",          "Mean rate over all events (Hz)"),
    ("participation_rate",      "Participation rate"),
    ("spikes_per_event_active", "Spikes per active event"),
]

fig, axes = plt.subplots(len(metrics), 2, figsize=(11, 9))
for row, (key, xlabel) in enumerate(metrics):
    for col, (df, label) in enumerate([(linear_replay, "Linear"),
                                       (open_replay,   "Open")]):
        ax  = axes[row, col]
        vals = df[key].dropna()
        ax.hist(vals, bins=40, color="0.6", edgecolor="white")
        ax.axvline(vals.median(), color="r", lw=2,
                   label=f"median={vals.median():.2f}")
        ax.axvline(vals.mean(),   color="b", lw=2, ls="--",
                   label=f"mean={vals.mean():.2f}")
        ax.set_xlabel(xlabel)
        ax.set_title(f"{label} | {key}")
        ax.legend(fontsize=8)
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "replay_rate_and_participation.png", dpi=300, bbox_inches="tight")
plt.show()


### gamma-phase modulation depth

Modulation depth is either defined here as $(max - min) / mean$ or Michelson $(max-min)/(max+min)$ of the phase-binned spike count curve. We do this on the **aggregated** step-level table from the decode pipeline (the .npy files).


In [ ]:
def modulation_depth(curve):
    c = curve[np.isfinite(curve)]
    if c.size == 0 or c.mean() == 0:
        return np.nan
    return (c.max() - c.min()) / c.mean()


def modulation_depth_michelson(curve):
    c = curve[np.isfinite(curve)]
    if c.size == 0 or (c.max() + c.min()) == 0:
        return np.nan
    return (c.max() - c.min()) / (c.max() + c.min())


def fit_cosine(binned, bin_width_deg=10.0):
    th = np.deg2rad((np.arange(len(binned)) + 0.5) * bin_width_deg)
    ok = np.isfinite(binned) # drop empty bins
    X  = np.column_stack([np.ones(ok.sum()), np.cos(th[ok]), np.sin(th[ok])])
    C, a, b = np.linalg.lstsq(X, np.asarray(binned)[ok], rcond=None)[0]
    return np.hypot(a, b) / C, np.degrees(np.arctan2(b, a)) % 360   # depth_m, pref_phase_deg


fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, data, label in [(axes[0], linear_data, "Linear"),
                        (axes[1], openfield_data, "Openfield")]:
    if len(data) == 0:
        ax.set_title(f"{label}: no data")
        continue

    curve = phase_binned_mean(data, "spikes")
    #md = modulation_depth(curve)
    md_michelson = modulation_depth_michelson(curve)

    # plot
    centers = np.arange(36) * 10 + 5
    ax.bar(centers, curve, width=10, color="0.7", edgecolor="white")
    ax.axhline(curve.mean(), color="k", lw=1, ls="--")
    ax.set_title(f"{label} spikes vs $\\gamma$ phase\nmodulation depth = {md_michelson*100:.2f}%")
    ax.set_xlabel("Gamma phase (deg)")
    ax.set_xlim(0, 360)

axes[0].set_ylabel("Mean spikes per step")
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "spike_modulation_depth.png", dpi=300, bbox_inches="tight")
plt.show()

print("Linear Track:")
print(linear_data["spikes"].describe())

print("Open Field:")
print(openfield_data["spikes"].describe())

depth, pref = fit_cosine(phase_binned_mean(linear_data, "2Dsteps"))
print(f"Linear Track - Cosine Fit: Depth = {depth*100:.2f}%, Preferred Phase = {pref:.2f}°")

### Place-field width with gaussian fits

Find a fit for each place field with either a 1d (linear track) or 2d (open field) Gaussian. For the 2d case, the function is:

Linear track:
$$ f(x) = A \exp\left(-\frac{(x - x_0)^2}{\sigma_x^2}\right) $$

Open field:
$$ f(x, y) = A \exp\left(-\left(\frac{(x - x_0)^2}{\sigma_x^2} + \frac{(y - y_0)^2}{\sigma_y^2}\right)\right) $$

Then $\sigma_x$ and $\sigma_y$ are the place field widths in the x and y dimensions, respectively. We can also compute an overall width as $\sqrt{\sigma_x^2 + \sigma_y^2}$. Parameters are for optimization bounds are below:

In [ ]:
MIN_PEAK_HZ    = 2.0
MIN_R2         = 0.3  # minimum of goodness of fit for place field to be included in width analysis
SIGMA_RANGE_CM = (1.0, 100.0)


def gauss2d(coords, A, x0, y0, sx, sy, off):
    x, y = coords
    return off + A * np.exp(-((x - x0) ** 2 / (sx ** 2)
                              + (y - y0) ** 2 / (sy ** 2)))

def gauss1d(x, A, x0, sigma, off):
    return off + A * np.exp(-(x - x0) ** 2 / (sigma ** 2))


#### Linear track fits

In [ ]:
def fit_field_1d(
    field, 
    bin_cm=BIN_CM,
    sigma_init=6.0,
):

    """
    Fit a 1-D Gaussian to the along-track (Y) marginal of a linear-track place field. The track runs along Y; X is just the corridor width and should carry no tuning information (TODO: Ask Lorenz if this is a reasonable assumption). 
    We collapse over X with nanmax (the peak rate at each Y position) and fit a 1-D Gaussian to that profile.
    """
    
    profile = np.nanmax(field, axis=1).astype(float)
    valid   = np.isfinite(profile)
    if not np.any(valid & (profile > 0)):
        return None
    peak = float(np.nanmax(profile))
    if peak < MIN_PEAK_HZ:
        return None
    
    # intial parameters and bounds for curve fitting
    ny     = profile.size
    y      = np.arange(ny) * bin_cm
    iy     = int(np.nanargmax(profile))
    p_init = (peak, iy * bin_cm, sigma_init, 0.0)
    bounds = ([0.1,      0,           SIGMA_RANGE_CM[0], -peak],
              [peak * 3, ny * bin_cm, SIGMA_RANGE_CM[1],  peak])

    # fit
    try:
        popt, _ = curve_fit(gauss1d, y[valid], profile[valid],
                            p0=p_init, bounds=bounds, maxfev=4000)
    except Exception as e:
        print(f"  fit failed: {e}")
        return None

    # return parameters and goodness of fit
    A, y0, sy, off = popt
    fit  = gauss1d(y[valid], *popt)    
    ss_r = np.sum((profile[valid] - fit) ** 2)
    ss_t = np.sum((profile[valid] - profile[valid].mean()) ** 2)
    r2   = 1 - ss_r / ss_t if ss_t > 0 else np.nan
    return dict(A=A, y0=y0, sy=sy, offset=off, r2=r2)
    
# fit place fields
out = []
for r in linear_fields:
    fit = fit_field_1d(r["field"])
    
    # if fit isn't good enough skip!
    if fit is None or not np.isfinite(fit["r2"]) or fit["r2"] < MIN_R2:
        continue
    
    out.append(dict(
        rat=r["rat"],
        session=r["session"],
        neuron=r["neuron"],
        sigma_cm=fit["sy"], 
        sy=fit["sy"], 
        r2=fit["r2"], 
        peak=r["peak_Hz"],
        A=fit["A"], 
        y0=fit["y0"], 
        offset=fit["offset"],
        field=r["field"]
    ))

linear_pfs_fits = pd.DataFrame(out)
del out

print(linear_pfs_fits.describe())
# save fits to parquet
if SAVE_STATS:
    linear_pfs_fits.to_parquet("linear_place_field_fits.parquet", index=False)

In [ ]:
# plot distribution of place field widths
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(linear_pfs_fits["sigma_cm"], bins=30, color="0.7", edgecolor="white")
ax.set_xlabel("Place field width (sigma, cm)")
ax.set_title(rf"Linear track place field widths (goodness of fit $R^2 > {MIN_R2}$)")
# stats
ax.axvline(linear_pfs_fits["sigma_cm"].median(), color="r", lw=2,
           label=f"median={linear_pfs_fits['sigma_cm'].median():.2f} cm")
ax.axvline(linear_pfs_fits["sigma_cm"].mean(), color="b", lw=2, ls="--",
           label=f"mean={linear_pfs_fits['sigma_cm'].mean():.2f} cm")
ax.legend()
fig.tight_layout()

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_place_field_widths.png", dpi=300, bbox_inches="tight")

plt.show()


In [ ]:
def show_example_fits_linear(df, n=6, seed=0):
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
    for k, (_, rec) in enumerate(df.sample(n=n, random_state=seed).iterrows()):
        profile = np.nanmax(rec["field"], axis=1)
        y       = np.arange(profile.size) * BIN_CM
        recon   = gauss1d(y, rec["A"], rec["y0"], rec["sy"], rec["offset"])
        axes[k].plot(y, profile, "k.", ms=4, label="data")
        axes[k].plot(y, recon,   "r-", lw=1.5, label="fit")
        axes[k].set_title(f"r{rec['rat']} {rec['session']}\n"
                          f"$\\sigma_y$={rec['sy']:.1f}  $R^2$={rec['r2']:.2f}")
        axes[k].set_xlabel("Y (cm)")
        # make x axis the same for every plot to make sigma comparable
        axes[k].set_xlim(0, 200)
        
    axes[0].set_ylabel("Peak rate over X (Hz)")
    fig.tight_layout()

show_example_fits_linear(linear_pfs_fits, n=5)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "example_linear_pf_fits.png", dpi=300, bbox_inches="tight")
plt.show()

#### Open field fits

Same as linear track, but with an additional $\sigma_y$ parameter that has the same bounds as $\sigma_x$.

In [ ]:
def fit_field(
    field, 
    bin_cm=BIN_CM,
    sigma_init=6.0,
):
    """
    Fit a 2-D Gaussian to a place field. Returns dict or None on failure.
    """
    
    ny, nx = field.shape
    yy, xx = np.mgrid[0:ny, 0:nx].astype(float) * bin_cm
    z      = field.astype(float)
    peak   = np.nanmax(z)
    if not np.isfinite(peak) or peak < MIN_PEAK_HZ:
        return None
    
    # intial parameters and bounds for curve fitting
    iy, ix = np.unravel_index(np.nanargmax(z), z.shape)
    p0     = (peak, ix * bin_cm, iy * bin_cm, sigma_init, sigma_init, 0.0)
    bounds = ([0.1,      0,           0,           SIGMA_RANGE_CM[0], SIGMA_RANGE_CM[0], -peak],
              [peak * 3, nx * bin_cm, ny * bin_cm, SIGMA_RANGE_CM[1], SIGMA_RANGE_CM[1],  peak])
    
    # fit
    try:
        popt, _ = curve_fit(gauss2d, (xx.ravel(), yy.ravel()), z.ravel(),
                            p0=p0, bounds=bounds, maxfev=4000)
    except Exception:
        return None
    
    # return parameters and goodness of fit
    A, x0, y0, sx, sy, off = popt
    fit  = gauss2d((xx, yy), *popt)
    ss_r = np.sum((z - fit) ** 2)
    ss_t = np.sum((z - z.mean()) ** 2)
    r2   = 1 - ss_r / ss_t if ss_t > 0 else np.nan
    return dict(A=A, x0=x0, y0=y0, sx=sx, sy=sy, offset=off, r2=r2)


# fit open field place fields
out = []
for r in open_fields:
    fit = fit_field(r["field"])
    
    # if fit isn't good enough skip!
    if fit is None or not np.isfinite(fit["r2"]) or fit["r2"] < MIN_R2:
        continue
    
    out.append(dict(
        rat=r["rat"],
        session=r["session"],
        neuron=r["neuron"],
        sigma_cm=np.sqrt(fit["sx"] * fit["sy"]), 
        sx=fit["sx"], sy=fit["sy"],
        r2=fit["r2"], 
        peak=r["peak_Hz"],
        A=fit["A"], 
        x0=fit["x0"], 
        y0=fit["y0"], 
        offset=fit["offset"],
        field=r["field"]
    ))

openfield_pfs_fits = pd.DataFrame(out)
del out

print(openfield_pfs_fits.describe())
# save fits to parquet
if SAVE_STATS:
    openfield_pfs_fits.to_parquet("openfield_place_field_fits.parquet", index=False)

In [ ]:
# plot distribution of place field widths for open field
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(openfield_pfs_fits["sigma_cm"], bins=30, color="0.7", edgecolor="white")
ax.set_xlabel("Place field width (sigma, cm)")
ax.set_title(fr"Open field place field widths (goodness of fit $R^2 > {MIN_R2}$)")
# stats
ax.axvline(openfield_pfs_fits["sigma_cm"].median(), color="r", lw=2,
           label=f"median={openfield_pfs_fits['sigma_cm'].median():.2f} cm")
ax.axvline(openfield_pfs_fits["sigma_cm"].mean(), color="b", lw=2, ls="--",
           label=f"mean={openfield_pfs_fits['sigma_cm'].mean():.2f} cm")
ax.legend()
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_place_field_widths.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def show_example_fits_open(df, n, seed=0):
    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5))
    for k, (_, rec) in enumerate(df.sample(n=n, random_state=seed).iterrows()):
        z      = rec["field"]
        ny, nx = z.shape
        yy, xx = np.mgrid[0:ny, 0:nx].astype(float) * BIN_CM
        recon  = gauss2d((xx, yy), rec["A"], rec["x0"], rec["y0"],
                         rec["sx"], rec["sy"], rec["offset"])
        axes[0, k].imshow(z, origin="lower", cmap="inferno")
        axes[0, k].set_title(f"data\nrat {rec['rat']} {rec['session']}")
        axes[1, k].imshow(recon, origin="lower", cmap="inferno")
        axes[1, k].set_title(f"fit $\\sigma_x$={rec['sx']:.1f} $\\sigma_y$={rec['sy']:.1f}\n$R^2$={rec['r2']:.2f}")
        for ax in (axes[0, k], axes[1, k]):
            ax.set_xticks([]) 
            ax.set_yticks([])
    fig.tight_layout()
    
show_example_fits_open(openfield_pfs_fits, n=5)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "example_openfield_pf_fits.png", dpi=300, bbox_inches="tight")
plt.show()

#### Fit PF widths with Lognormal distribution

In [ ]:
plot_log_normal_fit(openfield_pfs_fits["sigma_cm"], "Open field | PF width per cell", x_label="PF width (sigma, cm)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_pf_width_distribution_log_normal_fit.png", dpi=300, bbox_inches="tight")
plt.show()

#### Difference in PF width

Doing the same analysis like open field for linear track data:

In [ ]:
out = []

for r in linear_fields:
    fit = fit_field(r["field"])
    
    # if fit isn't good enough skip!
    if fit is None or not np.isfinite(fit["r2"]) or fit["r2"] < MIN_R2:
        continue
    
    out.append(dict(
        rat=r["rat"],
        session=r["session"],
        neuron=r["neuron"],
        sigma_cm=np.sqrt(fit["sx"] * fit["sy"]), 
        sx=fit["sx"], sy=fit["sy"],
        r2=fit["r2"], 
        peak=r["peak_Hz"],
        A=fit["A"], 
        x0=fit["x0"], 
        y0=fit["y0"], 
        offset=fit["offset"],
        field=r["field"]
    ))

linear2D_pfs_fits = pd.DataFrame(out)
del out

print(linear2D_pfs_fits.describe())
# save fits to parquet
if SAVE_STATS:
    linear2D_pfs_fits.to_parquet("linear2D_place_field_fits.parquet", index=False)

In [ ]:
# plot distribution of place field widths for linear track (2D fits)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(linear2D_pfs_fits["sigma_cm"], bins=30, color="0.7", edgecolor="white")
ax.set_xlabel("Place field width (sigma, cm)")
ax.set_title(r"Linear track place field widths from 2D fits (goodness of fit $R^2 > 0.3$)")
# stats
ax.axvline(linear2D_pfs_fits["sigma_cm"].median(), color="r", lw=2,
           label=f"median={linear2D_pfs_fits['sigma_cm'].median():.2f} cm")
ax.axvline(linear2D_pfs_fits["sigma_cm"].mean(), color="b", lw=2, ls="--",
           label=f"mean={linear2D_pfs_fits['sigma_cm'].mean():.2f} cm")
ax.legend()
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_place_field_widths_2D_fits.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
show_example_fits_open(linear2D_pfs_fits, n=10)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "example_linear_pf_fits_2D.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_log_normal_fit(linear2D_pfs_fits["sigma_cm"], "Linear 2D | PF width per cell", x_label="PF width (sigma, cm)")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear2D_pf_width_distribution_log_normal_fit.png", dpi=300, bbox_inches="tight")
plt.show()

#### Place field overlap


In [ ]:
FIELD_THRESH = 0.2  # fraction of peak to use as threshold for binary mask in overlap analysis

def _binary_mask_raw(field, thresh_frac=FIELD_THRESH):
    """Threshold a raw rate map at `thresh_frac` x peak -> boolean mask."""
    peak = np.nanmax(field)
    if not np.isfinite(peak) or peak <= 0:
        return None
    f = np.where(np.isfinite(field), field, 0.0)
    return f >= thresh_frac * peak

def _overlap_metrics(mask_a, mask_b):
    """Return Dice and Jaccard between two boolean masks."""
    intersection = np.sum(mask_a & mask_b)
    size_a, size_b = mask_a.sum(), mask_b.sum()
    union = size_a + size_b - intersection
    dice    = 2 * intersection / (size_a + size_b) if (size_a + size_b) > 0 else np.nan
    jaccard = intersection / union if union > 0 else np.nan
    return dice, jaccard


In [ ]:
from itertools import combinations


def pairwise_overlap(df, thresh_frac=FIELD_THRESH):
    """
    For every pair of neurons recorded in the same session,
    compute overlap of their thresholded raw rate maps.

    Needed columns in df: ["rat", "session", "neuron", "field"] where "field" is the 2D rate map.
    """
    rows = []
    for (rat, sess), grp in df.groupby(["rat", "session"]):
        entries = grp.to_dict("records")
        masks = []
        for e in entries:
            m = _binary_mask_raw(e["field"], thresh_frac)
            if m is not None:
                masks.append((e["neuron"], m))

        for (n1, m1), (n2, m2) in combinations(masks, 2):
            # masks must share the same shape (same session → same arena)
            if m1.shape != m2.shape:
                continue
            dice, jaccard = _overlap_metrics(m1, m2)
            rows.append(dict(
                rat=rat, session=sess,
                neuron_a=n1, neuron_b=n2,
                dice=dice, jaccard=jaccard,
                area_a=int(m1.sum()), area_b=int(m2.sum()), 
                intersection=int(np.sum(m1 & m2)),
            ))
    return pd.DataFrame(rows)

overlap_PF_openfield = pairwise_overlap(openfield_pfs_fits)

summary = overlap_PF_openfield["dice"].agg(["mean", "median", "std", "count"])
summary[["mean", "median", "std"]] *= 100  # convert to percentage
print(summary.round(2))
print()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
# show both overlap metrics for open field
for ax, metric in zip(axes, ["dice", "jaccard"]):
    vals = overlap_PF_openfield[metric].dropna() * 100  # percentage
    ax.hist(vals, bins=30, color="0.7", edgecolor="white")
    ax.axvline(vals.median(), color="r", lw=2,
               label=f"median={vals.median():.2f}%")
    ax.axvline(vals.mean(),   color="b", lw=2, ls="--",
               label=f"mean={vals.mean():.2f}%")
    ax.set_xlabel(f"Pairwise place field {metric.capitalize()} (%)")
    ax.set_title(f"Open field | {metric.capitalize()} overlap of thresholded rate maps")
    ax.legend()
axes[0].set_ylabel("Count of neuron pairs")
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "openfield_place_field_overlap.png", dpi=300, bbox_inches="tight")
plt.show()

# Spearman Rank Test

For the linear track. Step size against the absolute phase and with scipy's spearmanr test. 

In [ ]:
from utils import _fold_phase_rad


SPEARMAN_STYLE = dict(
    # scatter
    marker_size=6,          # geom_point size
    marker_alpha=0.4,
    marker_color="black",
    n_plot=1000,            # max dots DRAWN per panel (None = draw all)
    # lm fit
    fit_color="#3366FF",    # ggplot geom_smooth(method="lm") default blue
    fit_lw=2,
    ci=95,                  # None = no confidence band (much faster)
    n_boot=1000,            # bootstrap resamples for the CI band
    # axes
    xlabel="Absolute Phase [rad]",
    grid_color="0.85",
    grid_lw=0.6,
    # text
    stat_fontsize=10,       # rho / p annotation
    label_fontsize=None,    # None -> matplotlib default
    tick_fontsize=None,
    title_fontsize=None,
    title_weight="bold",
    show_stats=True,
)
 
 
def plot_vs_abs_phase(
    data,
    value_col,
    ylabel=None,
    title=None,
    ax=None,
    do_abs_step=False,
    scale=1.0,
    ylim=None,
    rng=None,
    **style,
):
    """Scatter of `value_col` against folded gamma phase, with lm fit and
    Spearman rho/p. rho, p and the fit use all finite rows; `n_plot` thins the
    drawn dots only."""
    unknown = set(style) - set(SPEARMAN_STYLE)
    if unknown:
        raise TypeError(f"unknown style key(s): {sorted(unknown)}")
    st = {**SPEARMAN_STYLE, **style}
 
    x = _fold_phase_rad(np.asarray(data["phase"], dtype=float))
    y = np.asarray(data[value_col], dtype=float) * scale
    if do_abs_step:
        y = np.abs(y)
 
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
 
    rho, p = stats.spearmanr(x, y)                                   # full data
 
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
 
    # thin the DRAWN dots only (legibility); never touches the numbers
    n_plot = st["n_plot"]
    if n_plot is not None and x.size > n_plot:
        if rng is None:
            rng = np.random.default_rng(0)
        draw = rng.choice(x.size, size=n_plot, replace=False)
    else:
        draw = np.arange(x.size)
 
    ax.scatter(x[draw], y[draw], alpha=st["marker_alpha"], s=st["marker_size"],
               color=st["marker_color"], edgecolors="none", zorder=1)
    sns.regplot(x=x, y=y, ax=ax, scatter=False, ci=st["ci"], n_boot=st["n_boot"],
                line_kws=dict(color=st["fit_color"], linewidth=st["fit_lw"], zorder=3))
 
    ax.set_xlim(0, np.pi)
    ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi])
    ax.set_xticklabels(["0", "$\\pi/4$", "$\\pi/2$", "3$\\pi/4$", "$\\pi$"])
    ax.set_xlabel(st["xlabel"], fontsize=st["label_fontsize"])
    ax.set_ylabel(ylabel if ylabel is not None else value_col,
                  fontsize=st["label_fontsize"])
    if st["tick_fontsize"] is not None:
        ax.tick_params(labelsize=st["tick_fontsize"])
    if ylim is not None:
        ax.set_ylim(*ylim)
    if title is not None:
        ax.set_title(title, fontweight=st["title_weight"], fontsize=st["title_fontsize"])
    ax.grid(True, color=st["grid_color"], linewidth=st["grid_lw"])
 
    if st["show_stats"]:
        p_str = "< 0.001" if p < 1e-3 else f"= {p:.3f}"
        ax.text(
            0.05, 0.92,
            f"$\\rho = {rho:.2f},\\ p {p_str}$",
            transform=ax.transAxes, va="top", ha="left",
            fontsize=st["stat_fontsize"], color="black",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.7"),
        )
 
    return ax


In [ ]:
plot_vs_abs_phase(
    linear_data,
    value_col="spikes",
    ylabel="Spikes",
    title="Linear: Spikes vs. absolute phase",
)

plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_spikes_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_vs_abs_phase(
    linear_data,
    value_col="2Dsteps",
    ylabel="2D step size",
    title="Linear: 2Dsteps vs. absolute phase",
)

plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_2dsteps_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_vs_abs_phase(
    linear_data,
    value_col="1Dsteps_x",
    ylabel="x step size",
    title="Linear: 1Dsteps_x vs. absolute phase",
)

plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_1dsteps_x_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plot_vs_abs_phase(
    linear_data,
    value_col="1Dsteps_y",
    ylabel="y step size",
    title="Linear: 1Dsteps_y vs. absolute phase",
)

plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "linear_1dsteps_y_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

## Movement Direction Analysis

In [ ]:
def plot_session(session_dir):
    P = np.atleast_2d(np.asarray(loadmat(session_dir / "Position_Data.mat",
                                          squeeze_me=True)["Position_Data"], dtype=float))
    t, x, y = P[:, 0], P[:, 1], P[:, 2]
    fig, ax = plt.subplots(figsize=(5, 4))
    sc = ax.scatter(x, y, c=t, s=1, cmap="viridis", alpha=0.3)
    ax.set_aspect("equal"); ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)")
    ax.set_title(str(session_dir.relative_to(BASE_DIR)))
    fig.colorbar(sc, ax=ax, label="time (s)")
    fig.tight_layout()
    fig.savefig(session_dir / "track_time_gradient.png", dpi=120)
    plt.show(fig)

for _, _, d in iter_sessions(LINEAR_FOLDERS):
    plot_session(d)

Label the direction of the event as forward or backward based on the sign of the y-step size. Then we can compare the phase distributions for forward vs backward events with the same analysis as before.

In [ ]:
grouped = linear_data.groupby("event_uid")
event_step_stats = pd.DataFrame({
    "net_dy": grouped["1Dsteps_y"].sum(),
    "consistency": grouped["1Dsteps_y"].apply(lambda x: np.sign(x).mean()),  # +1 if all steps are forward, -1 if all backward, 0 if half and half
    "n_steps": grouped.size(),
    "spike_count": grouped["spikes"].sum(),
})

print(event_step_stats.describe())

fig, ax = plt.subplots(1, 3, figsize=(10, 3))
ax[0].hist(event_step_stats["consistency"], bins=40); ax[0].set_xlabel("step consistency (mean sign of steps in event)"); ax[0].set_ylabel("count of events")
ax[1].hist(event_step_stats["net_dy"], bins=40);      ax[1].set_xlabel("net dy (decoder bins)")
# spike count per event
ax[2].hist(event_step_stats["spike_count"], bins=40); ax[2].set_xlabel("spike count per event")

plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "event_step_stats.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# mean spike count per event id (plot)
mean_spike_count_per_event = event_step_stats.groupby("event_uid")[["spike_count", "n_steps"]].mean()
print(mean_spike_count_per_event.describe())

# print the 20 events with the lowest mean spike count
print(mean_spike_count_per_event.nsmallest(40, "spike_count"))



In [ ]:
# plot consistency vs net_dy
fig, ax = plt.subplots(figsize=(6, 5))
mask = event_step_stats["n_steps"] >= 2
x = event_step_stats.loc[mask, "net_dy"]
y = event_step_stats.loc[mask, "consistency"]
ax.scatter(x, y, alpha=0.7)
ax.set_xlabel("Net Y displacement (cm)")
ax.set_ylabel("Consistency of step directions")
ax.grid(True, linestyle="--", alpha=0.2)
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "replay_event_consistency_vs_net_dy.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
event_step_stats["consistency_category"] = np.sign(event_step_stats["net_dy"]).astype(int)

# plot distribution of consistency categories
fig, ax = plt.subplots(figsize=(6, 4))
counts = event_step_stats["consistency_category"].value_counts().sort_index()
ax.bar(counts.index, counts.values, color="0.7", edgecolor="white")
ax.set_xticks([-1, 0, 1])
ax.set_xticklabels(["Backward", "Mixed", "Forward"])
ax.set_xlabel("Consistency category")
ax.set_ylabel("Count of events")
ax.set_title("Consistency of step directions within replay events")
fig.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "replay_event_consistency_category_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# print the number of events in each consistency category
print(counts)

In [ ]:
# distribution over n_steps for each consistency category
cat_map = {-1: "Backward", 0: "Mixed", 1: "Forward"}
order = ["Backward", "Mixed", "Forward"]
event_step_stats["consistency_label"] = event_step_stats["consistency_category"].map(cat_map)

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x="consistency_label", y="n_steps", data=event_step_stats,
            order=order, color="0.7", ax=ax)
ax.set_xlabel("Consistency category")
ax.set_ylabel("Number of steps in event")
ax.set_title("Number of steps in replay events by consistency category")
fig.tight_layout()
plt.show()

# print the mean and median number of steps in each consistency category
print(event_step_stats.groupby("consistency_label")["n_steps"].agg(["mean", "median", "count"]))


In [ ]:
# distribution over spike count for each consistency category
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x="consistency_label", y="spike_count", data=event_step_stats,
            order=order, color="0.7", ax=ax)
ax.set_xlabel("Consistency category")
ax.set_ylabel("Spike count")
ax.set_title("Spike count in replay events by consistency category")
fig.tight_layout()
plt.show()

In [ ]:
# select backward events
backward_events = event_step_stats[(event_step_stats["consistency_category"] < 0)]
print(backward_events.describe())



In [ ]:

plot_vs_abs_phase(
    linear_data[linear_data["event_uid"].isin(backward_events.index)],
    value_col="1Dsteps_y",
    ylabel="Y step size",
    title="Backward replay events: Y step size vs. absolute phase",
)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "backward_replay_1dsteps_y_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

plot_vs_abs_phase(
    linear_data[linear_data["event_uid"].isin(backward_events.index)],
    value_col="1Dsteps_x",
    ylabel="X step size",
    title="Backward replay events: X step size vs. absolute phase",
)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "backward_replay_1dsteps_x_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

plot_vs_abs_phase(
    linear_data[linear_data["event_uid"].isin(backward_events.index)],
    value_col="2Dsteps",
    ylabel="2D step size",
    title="Backward replay events: 2D step size vs. absolute phase",
)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "backward_replay_2dsteps_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# select forward events
forward_events = event_step_stats[event_step_stats["consistency_category"] > 0]
print(forward_events.describe())


In [ ]:
plot_vs_abs_phase(
    linear_data[linear_data["event_uid"].isin(forward_events.index)],
    value_col="1Dsteps_y",
    ylabel="Y step size",
    title="Forward replay events: Y step size vs. absolute phase",
)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "forward_replay_1dsteps_y_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

plot_vs_abs_phase(
    linear_data[linear_data["event_uid"].isin(forward_events.index)],
    value_col="1Dsteps_x",
    ylabel="X step size",
    title="Forward replay events: X step size vs. absolute phase",
)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "forward_replay_1dsteps_x_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

plot_vs_abs_phase(
    linear_data[linear_data["event_uid"].isin(forward_events.index)],
    value_col="2Dsteps",
    ylabel="2D step size",
    title="Forward replay events: 2D step size vs. absolute phase",
)
plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "forward_replay_2dsteps_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# doing the same analysis but with consistency_category * dy on linear_data
# this way we can look at all steps, not just those in events that are purely forward or backward

event_dir = np.sign(event_step_stats["net_dy"])  # +1 / -1 per event
linear_data["event_dir"]   = linear_data["event_uid"].map(event_dir)
linear_data["directed_dy"] = linear_data["event_dir"] * linear_data["1Dsteps_y"]
plot_vs_abs_phase(
    linear_data,
    value_col="directed_dy",
    ylabel="Directed Y step size",
    title="All replay events: Directed Y step size vs. absolute phase",
)

print(linear_data.describe())

plt.tight_layout()
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "all_replay_directed_dy_vs_abs_phase.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from utils import plot_spearman_grid

backward_mask = linear_data["event_uid"].isin(backward_events.index)
forward_mask = linear_data["event_uid"].isin(forward_events.index)

panels = [
    dict(
        data=linear_data,
        value_col="2Dsteps",
        ylabel="2D step size [cm]",
        title="2D (All events)",
        ylim=(0, 40),  # set per panel, or omit/None to autoscale
    ),
    dict(
        data=linear_data,
        value_col="1Dsteps_y",
        ylabel="dy [cm]",
        title="1D (All events)",
        ylim=(-40, 40),
    ),
    dict(
        data=linear_data,
        value_col="directed_dy",
        ylabel="directed dy [cm]",
        title="Directed 1D (All events)",
        ylim=(-40, 40),
    ),
    dict(
        data=linear_data[backward_mask],
        value_col="2Dsteps",
        ylabel="2D step size [cm]",
        title="2D (Backward events)",
        ylim=(0, 40),
    ),
    dict(
        data=linear_data[backward_mask],
        value_col="1Dsteps_y",
        ylabel="dy [cm]",
        title="1D (Backward events)",
        ylim=(-40, 40),
    ),
    None,   # directed dy == -dy within backward-only events
    dict(
        data=linear_data[forward_mask],
        value_col="2Dsteps",
        ylabel="2D step size [cm]",
        title="2D (Forward events)",
        ylim=(0, 40),
    ),
    dict(
        data=linear_data[forward_mask],
        value_col="1Dsteps_y",
        ylabel="dy [cm]",
        title="1D (Forward events)",
        ylim=(-40, 40),
    ),
    None,   # directed dy == dy within forward-only events
]

fig, axes = plot_spearman_grid(
    panels,
    plot_fn=plot_vs_abs_phase,
    ncols=3,
    sharey=False,
    save_path="figs_pf_analysis/spearman_panels_data_combined.png",
    figsize=(14, 8),
)
plt.show()


In [ ]:

panels = [
    dict(
        data=linear_data,
        value_col="2Dsteps",
        ylabel="2D step size [cm]",
        title="2D",
        ylim=(0, 30),
    ),
    dict(
        data=linear_data,
        value_col="directed_dy",
        ylabel="directed dy [cm]",
        title="Directed 1D",
        ylim=(-10, 25),
    ),
]

fig, axes = plot_spearman_grid(
    panels,
    plot_fn=plot_vs_abs_phase,
    ncols=2,
    sharey=False,
    save_path="figs_pf_analysis/spearman_panels_data_results.png",
    figsize=(10, 2.5),
)
plt.show()


In [ ]:
from utils import stepsize_2d_vs_1d_diff

summary_diff, boot_diff, ev_df = stepsize_2d_vs_1d_diff(
    df=linear_data,
    event_col="event_uid",
    phase_col="phase",
    two_step_col="2Dsteps",
    signed_y_col="1Dsteps_y", 
    n_boot=5000,
    seed=0
)

print("Summary:")
print(summary_diff)

print("\nBootstrap confidence intervals:")
print(boot_diff)


In [ ]:
from utils import plot_stepsize_2d_vs_1d_diff

fig = plot_stepsize_2d_vs_1d_diff(summary_diff, boot_diff, whisker_ymax=400)

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "stepsize_2d_vs_1d_diff.png", dpi=300, bbox_inches="tight")
plt.show(fig)


In [ ]:
print("Event-level spike count statistics:")
print(linear_data.groupby("event_uid")["spikes"].sum().describe())


print("\nOverall spike count statistics:")
print(linear_data["spikes"].describe())    

In [ ]:
from utils import stepsize_noise_slope, plot_stepsize_noise_slope, plot_stepsize_noise_slope_bootstrap

summary_slope, boot_slope, per_step = stepsize_noise_slope(linear_data, n_boot=5000, min_spikes=2, n_lo_pct=97.5, n_hi_pct=2.5, seed=0)
print(summary_slope)  # inflation_2D, inflation_directed_1D, difference cm, with CIs
print(summary_slope.attrs)  # slopes, intercepts, N_lo/N_hi, n_steps, p_diff_le_0

print(per_step.describe())

In [ ]:


fig = plot_stepsize_noise_slope(summary_slope, per_step, fit_form="inv_sqrt_n", figsize=(8, 4), n_bins=8, scatter=5000, ylim=(0, 40), xlim=(0, 50))

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "stepsize_noise_slope.png", dpi=300, bbox_inches="tight")
plt.show(fig)

In [ ]:
# doing the same analysis with open field data
summary_slope_open, boot_slope_open, per_step_open = stepsize_noise_slope(openfield_data, n_boot=5000, min_spikes=2, n_lo_pct=97.5, n_hi_pct=2.5, seed=0)
print(summary_slope_open)  #inflation_2D, inflation_directed_1D, difference cm, with CIs

print(per_step_open.describe())

# number of events in open field data
print("Number of events in open field data:", openfield_data["event_uid"].nunique())

fig = plot_stepsize_noise_slope(summary_slope_open, per_step_open, fit_form="inv_sqrt_n", figsize=(11, 3), n_bins=8, add_1D=False, scatter=5000, ylim=(0, 40), xlim=(0, 50),
title=None)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "stepsize_noise_slope_openfield.png", dpi=300, bbox_inches="tight")
plt.show(fig)

In [ ]:
from utils import stepsize_threshold_prob, plot_stepsize_threshold_prob


summ_2d, boot_2d = stepsize_threshold_prob(per_step, measure="step_2d_cm",n_bins=8, thresh_less=4, thresh_greater=5)
fig = plot_stepsize_threshold_prob(summ_2d, title=None)

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "2D_stepsize_threshold_prob.png", dpi=300, bbox_inches="tight")
plt.show(fig)

In [ ]:
summ_directed, boot_directed = stepsize_threshold_prob(per_step,n_bins=8, measure="directed_1d_cm", thresh_less=4, thresh_greater=5)
fig = plot_stepsize_threshold_prob(summ_directed, title=None)

if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "directed_1D_stepsize_threshold_prob.png", dpi=300, bbox_inches="tight")
plt.show(fig)

In [ ]:
panels = [
    dict(summary=summ_2d, title="2D step size"),
    dict(summary=summ_directed, title="1D directed step size"),
]
 
fig, axes = plot_spearman_grid(
    panels,
    plot_fn=plot_stepsize_threshold_prob,
    ncols=2,
    sharey=True,
    save_path="prob_plot.png",
    figsize=(9, 4),
)
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "prob_plot_combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from utils import plot_stepsize_combined

fig = plot_stepsize_combined(summary_diff, boot_diff, summary_slope, per_step, 
                             n_bins=8, figsize=(11, 6.5), whisker_ymax=420, fit_form="inv_sqrt_n")
if SAVE_FIGS:
    plt.savefig(SAVE_FIGS / "stepsize_combined.png", dpi=300, bbox_inches="tight")
plt.show(fig)

In [ ]:

from pathlib import Path

SIM_STEPDATA = Path(
    "../python/data/"
    "percellrates_sim_seq_mode-smooth_paramsid_16_250reps_1mincells_200cells_25steps_0.21gammamod_2.6meanHz_stepdata.npy"
)

YLIM = (0, 27)   # shared across both panels
FSIZE = 10
 
SKIP_THIS_CELL = True

def load_sim_stepdata(path):
    """step_spikes / step_phase -> frame with the notebook's column names."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"sim sidecar not found: {path.resolve()}")
    arr = np.load(path, allow_pickle=True)
    df = pd.DataFrame({"phase": np.asarray(arr[1], float).ravel(),
                       "spikes": np.asarray(arr[0], float).ravel()})
    n_drop = int(df.isna().any(axis=1).sum())
    df = df.dropna().reset_index(drop=True)
    n_uniq = np.unique(np.round(df["phase"].to_numpy(), 3)).size
    print(f"[sim] {path.name}")
    print(f"[sim] {len(df)} steps (dropped {n_drop} NaN), {n_uniq} distinct phase values")
    return df
 
 
def _smooth(binned, window=8):
    """Boxcar with wrap-around at 0/360; NaN bins masked out, not propagated."""
    n = len(binned)
    v = np.tile(np.nan_to_num(binned, nan=0.0), 3)
    m = np.tile(np.isfinite(binned).astype(float), 3)
    k = np.ones(window) / window
    num, den = np.convolve(v, k, mode="same"), np.convolve(m, k, mode="same")
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)[n:2 * n]
 
def _panel_letter(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontweight="bold",
            fontsize=FSIZE + 4, va="bottom", ha="right")


def lognorm_panel(ax, values, title, x_label):
    """Fit + plot a log-normal distribution on a given axes."""
    ax.hist(values, bins=60, density=True, color="0.7", edgecolor="white")

    shape, loc, scale = stats.lognorm.fit(values, floc=0)
    x = np.linspace(values.min(), values.max(), 500)
    pdf = stats.lognorm.pdf(x, shape, loc=loc, scale=scale)
    ax.plot(x, pdf, color="r", linewidth=2,
            label=f"LogNorm fit \n($\\sigma={shape:.2f}$, \n $\\mu={np.log(scale):.2f}$, \n $median={scale:.2f}$)")

    ax.set_xlabel(x_label, fontsize=FSIZE)
    ax.set_ylabel("density", fontsize=FSIZE)
    ax.legend(loc="upper right", fontsize=FSIZE - 2)
    ax.set_title(title, fontsize=FSIZE)
    ax.tick_params(labelsize=FSIZE)


def plot_data_vs_sim(data_df, sim_df, n_bins=36, bin_width_deg=10,
                      labels=("Data", "Simulation"), letters=("a", "b"),
                      ylim=YLIM, figsize=(7, 3.0), axes=None):
    if axes is None:
        fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True,
                                  dpi=300, constrained_layout=True)

    centres = (np.arange(n_bins) + 0.5) * bin_width_deg

    for ax, df, lab, letter in zip(axes, (data_df, sim_df), labels, letters):
        curve = phase_binned_mean(df, "spikes", n_bins=n_bins)
        depth = modulation_depth_michelson(curve)   # same metric as the sim script

        ax.bar(centres, curve, width=bin_width_deg, color="0.7", edgecolor="white")
        ax.plot(centres, _smooth(curve), color="r", lw=3, solid_capstyle="round")
        ax.axhline(np.nanmean(curve), color="k", lw=1, ls="--", zorder=0)
        ax.set_ylim(ylim)
        ax.set_xlabel("Absolute phase [deg]", fontsize=FSIZE)
        ax.set_title(f"{lab} | depth {depth*100:.1f}%", fontsize=FSIZE)
        for t in ax.get_xticklabels() + ax.get_yticklabels():
            t.set_fontsize(FSIZE)
        _panel_letter(ax, letter)
        print(f"[{letter}] {lab}: {len(df)} steps, mean {np.nanmean(curve):.2f} spikes/step, "
              f"depth {depth*100:.2f}%")

    axes[0].set_ylabel("Mean spikes per step", fontsize=FSIZE)
    axes[1].tick_params(labelleft=False)   # shares ylim with the panel to its left
    return axes[0].figure, axes


if not SKIP_THIS_CELL:
    sim_data = load_sim_stepdata(SIM_STEPDATA)

    fig, axes = plt.subplots(1, 4, figsize=(13, 3), dpi=300, constrained_layout=True)

    lognorm_panel(axes[0], open_peaks, "Peak firing rate per cell", "Peak firing rate [Hz]")
    lognorm_panel(axes[1], openfield_pfs_fits["sigma_cm"], "PF width per cell", r"PF width $\sigma$ [cm]")
    _panel_letter(axes[0], "a")
    _panel_letter(axes[1], "b")

    plot_data_vs_sim(linear_data, sim_data, letters=("c", "d"), axes=axes[2:])

    if SAVE_FIGS:
        fig.savefig(SAVE_FIGS / "appendix_data_vs_sim_phase.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
linear_data["n_shifts"] = np.round(linear_data["dt"] / DECODE_STEP).astype(int)

win_linear_data["peak_prob_z"] = win_linear_data.groupby(["rat", "session"])["peak_prob"].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) > 0 else s * np.nan)

def report_dt(data, save_figs=SAVE_FIGS):
    n = data["n_shifts"].to_numpy()
    counts = pd.Series(n).value_counts().sort_index()
    frac = counts / counts.sum()

    print(f"--- P2: step duration (DECODE_STEP = {DECODE_STEP*1e3:.0f} ms) ---")
    for k, c in counts.items():
        print(f"{k:8d}  {k*DECODE_STEP*1e3:5.0f} ms  {c:8d}  {frac[k]:7.4f}")
    print(f"fraction with dt == {DECODE_STEP*1e3:.0f} ms : {frac.get(1, 0.0):.4f}")
    print(f"median {np.median(n)*DECODE_STEP*1e3:.1f} ms | mean {np.mean(n)*DECODE_STEP*1e3:.2f} ms | max {n.max()*DECODE_STEP*1e3:.0f} ms")

    phase  = _fold_phase_rad(data["phase"].to_numpy())
    step2d = data["2Dsteps"].to_numpy() * BIN_CM
    spikes = data["spikes"].to_numpy()
    dt     = data["dt"].to_numpy()
    rate   = np.where(dt > 0, spikes / dt, np.nan)

    print("\nSpearman rho with dt (pooled, descriptive only):")
    for name, v in [("2D step size", step2d), ("spike count", spikes),
                    ("spike rate", rate), ("folded phase", phase)]:
        m = np.isfinite(dt) & np.isfinite(v)
        r, p = stats.spearmanr(dt[m], v[m])
        print(f"  dt vs {name:14s}  rho = {r:+.3f}  p = {p:.2g}  (n = {m.sum()})")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].bar(counts.index * DECODE_STEP * 1e3, counts.values,
              width=DECODE_STEP * 1e3 * 0.8, color="0.6", edgecolor="white")
    ax[0].set_xlabel("Step duration [ms]"); ax[0].set_ylabel("Count of steps"); ax[0].set_yscale("log")

    ks = [k for k in sorted(counts.index) if k <= 6 and counts[k] > 5]
    ax[1].boxplot([step2d[n == k] for k in ks], positions=ks, widths=0.6, showfliers=False)
    ax[1].set_xlabel("Step duration [n x 5 ms]"); ax[1].set_ylabel("2D step size [cm]")
    ax[1].grid(alpha=0.3, ls="--")
    fig.tight_layout()
    if save_figs:
        fig.savefig(save_figs / "p2_step_duration.png", dpi=300, bbox_inches="tight")
    plt.show()

report_dt(linear_data)

In [ ]:
def _session_phase_curves(win, value_col, n_bins=36):
    edges = np.linspace(0, 360, n_bins + 1)
    curves = []
    for _, g in win.groupby(["rat", "session"]):
        v = g[value_col].to_numpy(dtype=float)
        idx = np.clip(np.digitize(g["phase"].to_numpy(), edges[1:-1]), 0, n_bins - 1)
        c = np.array([np.nanmean(v[idx == k]) if (idx == k).any() else np.nan
                      for k in range(n_bins)])
        curves.append(c)
    return np.vstack(curves)


def _running_box_circular(curve, box=8):
    n = curve.size
    out = np.full(n, np.nan)
    half = box // 2
    for i in range(n):
        idx = [(i + k - half) % n for k in range(box)]
        vals = curve[idx]
        if np.isfinite(vals).any():
            out[i] = np.nanmean(vals)
    return out


def plot_peakprob_histogram(win, n_bins=36, box=8, figsize=(6, 4), ax=None):
    curves = _session_phase_curves(win, "peak_prob", n_bins=n_bins)
    mean_c = np.nanmean(curves, axis=0)
    sem_c  = np.nanstd(curves, axis=0) / np.sqrt(np.sum(np.isfinite(curves), axis=0))
    box_c  = _running_box_circular(mean_c, box=box)

    centers = np.arange(n_bins) * (360 / n_bins) + (360 / n_bins) / 2
    x2 = np.concatenate([centers, centers + 360])
    y2, e2, b2 = np.tile(mean_c, 2), np.tile(sem_c, 2), np.tile(box_c, 2)

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    ax.bar(x2, y2, width=360 / n_bins, color="0.7", edgecolor="white")
    ax.errorbar(x2, y2, yerr=e2, fmt="none", ecolor="0.4", elinewidth=0.7)
    ax.plot(x2, b2, color="tab:red", lw=1.8)
    ax.set_xlim(10, 720)
    ax.set_xticks([10, 180, 360, 540, 720])
    ax.set_xlabel("Gamma phase [deg] for two cycles")
    ax.set_ylabel("Peak posterior probability")
    if standalone:
        fig.tight_layout()
    return fig, ax


def plot_peakprob_polar(win, steps, n_bins=36, box=8, figsize=(5.0, 5.2), ax=None, legend=True):
    centers_rad = np.deg2rad(np.arange(n_bins) * (360 / n_bins) + (180 / n_bins))

    st = steps.copy()
    st["step_cm"] = st["2Dsteps"] * BIN_CM

    series = [("peak_prob", win, "tab:red",   "Norm. peak posterior"),
              ("step_cm",   st,  "tab:blue",  "Norm. step size"),
              ("spikes",    st,  "tab:green", "Norm. population rate")]


    standalone = ax is None
    if standalone:
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111, projection="polar")
    else:
        fig = ax.figure

    means = {}
    for col, df, colour, label in series:
        curves = _session_phase_curves(df, col, n_bins=n_bins)
        c = _running_box_circular(np.nanmean(curves, axis=0), box=box)
        lo, hi = np.nanmin(c), np.nanmax(c)
        c = (c - lo) / (hi - lo) if hi > lo else c * np.nan
        ax.plot(np.append(centers_rad, centers_rad[0]), np.append(c, c[0]),
                color=colour, lw=1.6, label=label)
        w = np.nan_to_num(c)
        if not np.any(w):
            means[label] = np.nan
            continue
        
        mu = np.angle(np.sum(w * np.exp(1j * centers_rad))) % (2 * np.pi)
        means[label] = np.rad2deg(mu)
        ax.annotate("", xy=(mu, 1.0), xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color=colour, lw=1.8))

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1.05)
    ax.set_yticklabels([])
    ax.xaxis.grid(True, linestyle='--', linewidth=0.8) 
    ax.yaxis.grid()

    if legend:
        ax.legend(loc="right", bbox_to_anchor=(-.4, 0.5), frameon=True, ncol=1, fontsize=10)
    if standalone:
        fig.tight_layout()
    print("circular weighted means [deg]: " + ", ".join(f"{k} = {v:.0f}" for k, v in means.items()))
    return fig, ax


In [ ]:
plot_peakprob_histogram(win_linear_data)
plot_peakprob_polar(win_linear_data, linear_data, figsize=(10, 4), legend=True)

In [ ]:
plot_peakprob_histogram(win_openfield_data)
plot_peakprob_polar(win_openfield_data, openfield_data)

In [ ]:
def _add_panel_label(ax, label):
    ax.text(-0.18, 1.08, label, transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="bottom", ha="left")

def create_trajectory_figure_panel(
    linear_data, openfield_data,
    win_linear_data, win_openfield_data,
    figsize=(12, 12),
    ylims=None,
    n_bins=36,
    smooth_window=8,
    col_titles=("Linear track", "Open field"),
    panel_labels="abcdefgh",
    save_path=None,
):
    default_ylims = {"spikes": {"linear": None, "open": None},
                      "2Dsteps": {"linear": None, "open": None},
                      "peak_prob": {"linear": None, "open": None}}
    if ylims:
        for k, v in ylims.items():
            default_ylims.setdefault(k, {"linear": None, "open": None}).update(v)
    ylims = default_ylims

    columns = [
        ("linear", col_titles[0], linear_data, win_linear_data),
        ("open",   col_titles[1], openfield_data, win_openfield_data),
    ]

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(4, 2, hspace=0.6, wspace=0.35)
    labels = iter(panel_labels)

    row_specs = [("spikes", "spikes"), ("2Dsteps", "2Dsteps")]
    row_ylabels = {"spikes": "Spikes per bin", "2Dsteps": "Step size per bin [cm]"}   # hardcoded

    for row_i, (row_key, value_col) in enumerate(row_specs):
        for col_i, (col_key, col_title, step_df, win_df) in enumerate(columns):
            ax = fig.add_subplot(gs[row_i, col_i])
            plot_phase_panel(step_df, value_col=value_col, n_bins=n_bins,
                              smooth_window=smooth_window, ylim=ylims[row_key][col_key], ax=ax)
            ax.set_ylabel(row_ylabels[row_key])   # hardcoded override
            _add_panel_label(ax, next(labels))
            if row_i == 0:
                ax.set_title(col_title, fontsize=12, pad=18)

    # row 3: peak posterior
    for col_i, (col_key, col_title, step_df, win_df) in enumerate(columns):
        ax = fig.add_subplot(gs[2, col_i])
        plot_peakprob_histogram(win_df, n_bins=n_bins, box=smooth_window, ax=ax)
        ax.set_ylabel("Peak posterior probability")   # hardcoded override
        _add_panel_label(ax, next(labels))

    # row 4: polar
    for col_i, (col_key, col_title, step_df, win_df) in enumerate(columns):
        ax = fig.add_subplot(gs[3, col_i], projection="polar")
        plot_peakprob_polar(win_df, step_df, n_bins=n_bins, box=smooth_window, ax=ax,)
        _add_panel_label(ax, next(labels))

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
fig = create_trajectory_figure_panel(
    linear_data, openfield_data,
    win_linear_data, win_openfield_data,
    figsize=(11, 13),
    ylims={
        "spikes":  {"linear": (0, 22), "open": (0, 20)},
        "2Dsteps": {"linear": (0, 5),   "open": (0, 6.5)},
    },
    save_path=(SAVE_FIGS / "trajectory_phase_panel.png") if SAVE_FIGS else None,
)
plt.show()

In [ ]:
N_BINS = 36
BOX    = 8

st = linear_data.copy()
st["step2d_cm"] = st["2Dsteps"]     * BIN_CM
st["dirdy_cm"]  = st["directed_dy"] * BIN_CM

rows = [
    dict(value_col="2Dsteps",     polar_col="step2d_cm", colour="tab:orange",
         ylabel="2D step size [cm]", ylim=(0, 30), title="2D",
         polar_label="Norm. 2D step size"),
    dict(value_col="directed_dy", polar_col="dirdy_cm",  colour="tab:blue",
         ylabel="directed dy [cm]",  ylim=(-10, 20), title="Directed 1D",
         polar_label="Norm. directed 1D step size"),
]

centers_rad = np.deg2rad(np.arange(N_BINS) * (360 / N_BINS) + (180 / N_BINS))
theta = np.append(centers_rad, centers_rad[0])


def _curve(col):
    """session curves -> mean over sessions -> boxcar -> min-max to [0, 1]."""
    c = _running_box_circular(
        np.nanmean(_session_phase_curves(st, col, n_bins=N_BINS), axis=0), box=BOX)
    lo, hi = np.nanmin(c), np.nanmax(c)
    print(f"{col:<11s} pre-norm min {lo:6.2f} | max {hi:6.2f} | "
          f"range {hi - lo:5.2f} | mean {np.nanmean(c):6.2f}")
    return (c - lo) / (hi - lo) if hi > lo else c * np.nan


def _arrow(ax, c, colour):
    w = np.nan_to_num(c)
    if not np.any(w):
        return np.nan
    mu = np.angle(np.sum(w * np.exp(1j * centers_rad))) % (2 * np.pi)
    ax.annotate("", xy=(mu, 1.0), xytext=(0, 0),
                arrowprops=dict(arrowstyle="-|>", color=colour, lw=1.8))
    return np.rad2deg(mu)


rate = _curve("spikes")

fig = plt.figure(figsize=(12, 6.5), dpi=300, constrained_layout=True)
gs  = fig.add_gridspec(2, 2, width_ratios=[1.2, 1])
means, polar_axes = {}, []

for r, cfg in enumerate(rows):
    ax_s = fig.add_subplot(gs[r, 0])
    ax_p = fig.add_subplot(gs[r, 1], projection="polar")

    # a) / c) spearman scatter + lm fit
    plot_vs_abs_phase(linear_data, value_col=cfg["value_col"], scale=BIN_CM,
                      ylabel=cfg["ylabel"], title=cfg["title"], ylim=cfg["ylim"], ax=ax_s)

    # b) / d)
    c = _curve(cfg["polar_col"])
    ax_p.plot(theta, np.append(c, c[0]), color=cfg["colour"], lw=1.6, label=cfg["polar_label"])
    ax_p.plot(theta, np.append(rate, rate[0]), color="tab:green", lw=1.6,
              label="Norm. population rate")
    means[cfg["polar_label"]]      = _arrow(ax_p, c, cfg["colour"])
    means["Norm. population rate"] = _arrow(ax_p, rate, "tab:green")

    polar_axes.append(ax_p)
    ax_p.set_theta_zero_location("N")
    ax_p.set_theta_direction(-1)
    ax_p.xaxis.grid(True, linestyle='--', linewidth=0.8) 
    ax_p.yaxis.grid()
    ax_p.set_ylim(0, 1.05)
    ax_p.set_yticklabels([])
    ax_p.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), frameon=True, fontsize=9)

    for ax, letter, dx in ((ax_s, "ac"[r], -0.12), (ax_p, "bd"[r], -0.5)):
        ax.text(dx, 1.08, letter, transform=ax.transAxes, fontsize=16,
                fontweight="bold", va="top", ha="left")

fig.tight_layout(h_pad=3.0)

# 1.0 = no shrink
POLAR_SHRINK = 0.9
for ax in polar_axes:
    b = ax.get_position()
    ax.set_position([b.x0 + b.width  * (1 - POLAR_SHRINK) / 2,
                     b.y0 + b.height * (1 - POLAR_SHRINK) / 2,
                     b.width * POLAR_SHRINK, b.height * POLAR_SHRINK])

print("circular weighted means [deg]: " + ", ".join(f"{k} = {v:.0f}" for k, v in means.items()))
if SAVE_FIGS:
    fig.savefig(SAVE_FIGS / "spearman_and_polar_2d_vs_directed1d.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from utils import _prepare_steps, _ols_slope_intercept, COLORS

MIN_SPIKES = 2
HALF_WIDTH = 45 # deg
N_LIM = (2.0, 50.0) # spike counts shown
Y_LIM = (0.0, 20.0)
N_TICKS = [2, 3, 5, 8, 12, 20, 35, 50]
HIST_EDGES = np.arange(2, 52, 4)
COLOR = COLORS["step_2d_cm"]

WINDOWS = [
    dict(centre=355.0, style="-",  label=r"$355^\circ \pm 45^\circ$ (rate peak)"),
    dict(centre=165.0, style="--", label=r"$165^\circ \pm 45^\circ$ (2D-step peak)"),
]


def inv_sqrt(n):
    return 1.0 / np.sqrt(np.asarray(n, dtype=float))


def circular_distance(angles, centre):
    return np.abs((angles - centre + 180.0) % 360.0 - 180.0)


steps = _prepare_steps(
    linear_data, event_col="event_uid", spikes_col="spikes",
    two_step_col="2Dsteps", signed_y_col="1Dsteps_y",
    bin_size_cm=BIN_CM, min_spikes=MIN_SPIKES,
)
kept = ((linear_data["spikes"] >= MIN_SPIKES)
        & linear_data[["2Dsteps", "1Dsteps_y"]].notna().all(axis=1)).to_numpy()
assert kept.sum() == len(steps)
phase = linear_data.loc[kept, "phase"].to_numpy(dtype=float)

for window in WINDOWS:
    inside = np.isfinite(phase) & (circular_distance(phase, window["centre"]) <= HALF_WIDTH)
    subset = steps[inside].reset_index(drop=True)
    slope, intercept = _ols_slope_intercept(
        subset["inv_sqrt_n"].to_numpy(), subset["step_2d_cm"].to_numpy())
    window.update(steps=subset, slope=slope, intercept=intercept)


# fits are drawn only where both windows have data
grid = np.linspace(max(np.percentile(w["steps"]["spikes"], 2.5) for w in WINDOWS),
                   min(np.percentile(w["steps"]["spikes"], 97.5) for w in WINDOWS), 200)


def draw_fits(ax, x_of_n):
    for window in WINDOWS:
        y = window["intercept"] + window["slope"] * inv_sqrt(grid)
        ax.plot(x_of_n(grid), y, color=COLOR, ls=window["style"], lw=1.5, label=window["label"])
    ax.set_ylim(*Y_LIM)
    ax.set_ylabel("Mean 2D step size [cm]")
    ax.grid(alpha=0.3, ls="--")
    ax.legend(frameon=True, fontsize=8, title=r"$1/\sqrt{N}$ fit", title_fontsize=8)


def figure_vs_n():
    """x axis in N, with the two spike-count distributions above."""
    fig, ax = plt.subplots(figsize=(11, 4))
    marginal = ax.inset_axes([0, 1.0, 1, 0.18], sharex=ax)

    draw_fits(ax, x_of_n=lambda n: n)
    for window in WINDOWS:
        counts = window["steps"]["spikes"].to_numpy()
        marginal.hist(counts, bins=HIST_EDGES, histtype="step", color=COLOR,
                      ls=window["style"], lw=1.2,
                      weights=np.full(len(counts), 1.0 / len(counts)))

    ax.set_xlim(*N_LIM)
    ax.set_xlabel(r"Number of spikes per step $N$")
    marginal.set_ylabel("frac.", fontsize=7)
    marginal.tick_params(labelbottom=False, labelleft=False, left=False)
    for side in ("top", "right", "left"):
        marginal.spines[side].set_visible(False)
    fig.tight_layout()
    return fig


def figure_vs_inv_sqrt_n():
    """x axis linear in 1/sqrt(N), so both fits are straight lines."""
    fig, ax = plt.subplots(figsize=(11, 3))
    draw_fits(ax, x_of_n=inv_sqrt)

    ax.set_xlim(*inv_sqrt(np.array(N_LIM))[::-1])
    ax.set_xticks(inv_sqrt(N_TICKS))
    ax.set_xticklabels(N_TICKS)
    ax.set_xlabel(r"Number of spikes per step $N$ (axis linear in $1/\sqrt{N}$)")
    fig.tight_layout()
    return fig


fig_n, fig_inv = figure_vs_n(), figure_vs_inv_sqrt_n()
if SAVE_FIGS:
    fig_n.savefig(SAVE_FIGS / "stepsize_phase_overlay_N.png", dpi=300, bbox_inches="tight")
    fig_inv.savefig(SAVE_FIGS / "stepsize_phase_overlay_invsqrtN.png", dpi=300, bbox_inches="tight")
plt.show()


pooled_slope = _ols_slope_intercept(
    np.concatenate([w["steps"]["inv_sqrt_n"].to_numpy() for w in WINDOWS]),
    np.concatenate([w["steps"]["step_2d_cm"].to_numpy() for w in WINDOWS]))[0]

# printout some statistics
for window in WINDOWS:
    subset = window["steps"]
    print(f"{window['centre']:.0f} deg: {len(subset)} steps,"
          f"mean 2D step {subset['step_2d_cm'].mean():.2f} cm, "
          f"mean N {subset['spikes'].mean():.1f},"
          f"fit {window['intercept']:.2f} + {window['slope']:.2f}/sqrt(N)")

observed = (WINDOWS[1]["steps"]["step_2d_cm"].mean()
            - WINDOWS[0]["steps"]["step_2d_cm"].mean())
predicted = pooled_slope * (WINDOWS[1]["steps"]["inv_sqrt_n"].mean()
                            - WINDOWS[0]["steps"]["inv_sqrt_n"].mean())
print(f"observed difference   : {observed:+.2f} cm")


In [ ]:
DATA = openfield_data
SLUG = "openfield"

MIN_SPIKES = 2
HALF_WIDTH = 45              # deg
N_LIM = (2.0, 50.0)
Y_LIM = (0.0, 30.0)
N_TICKS = [2, 3, 5, 8, 12, 20, 35, 50]
HIST_EDGES = np.arange(2, 52, 4)
COLOR = COLORS["step_2d_cm"]

WINDOWS = [
    dict(centre=33.0,  style="-",  label=r"$33^\circ \pm 45^\circ$ (rate peak)"),
    dict(centre=172.0, style="--", label=r"$172^\circ \pm 45^\circ$ (2D-step peak)"),
]


assert circular_distance(WINDOWS[0]["centre"], WINDOWS[1]["centre"]) > 2 * HALF_WIDTH, \
    "phase windows overlap; steps would be counted twice"

steps = _prepare_steps(
    DATA, event_col="event_uid", spikes_col="spikes",
    two_step_col="2Dsteps", signed_y_col="1Dsteps_y",
    bin_size_cm=BIN_CM, min_spikes=MIN_SPIKES,
)
kept = ((DATA["spikes"] >= MIN_SPIKES)
        & DATA[["2Dsteps", "1Dsteps_y"]].notna().all(axis=1)).to_numpy()
assert kept.sum() == len(steps)
phase = DATA.loc[kept, "phase"].to_numpy(dtype=float)

for window in WINDOWS:
    inside = np.isfinite(phase) & (circular_distance(phase, window["centre"]) <= HALF_WIDTH)
    subset = steps[inside].reset_index(drop=True)
    slope, intercept = _ols_slope_intercept(
        subset["inv_sqrt_n"].to_numpy(), subset["step_2d_cm"].to_numpy())
    window.update(steps=subset, slope=slope, intercept=intercept)

# fits are drawn only where both windows have data
grid = np.linspace(max(np.percentile(w["steps"]["spikes"], 2.5) for w in WINDOWS),
                   min(np.percentile(w["steps"]["spikes"], 97.5) for w in WINDOWS), 200)


fig_n, fig_inv = figure_vs_n(), figure_vs_inv_sqrt_n()
if SAVE_FIGS:
    fig_n.savefig(SAVE_FIGS / f"stepsize_phase_overlay_{SLUG}_N.png", dpi=300, bbox_inches="tight")
    fig_inv.savefig(SAVE_FIGS / f"stepsize_phase_overlay_{SLUG}_invsqrtN.png", dpi=300, bbox_inches="tight")
plt.show()

pooled_slope = _ols_slope_intercept(
    np.concatenate([w["steps"]["inv_sqrt_n"].to_numpy() for w in WINDOWS]),
    np.concatenate([w["steps"]["step_2d_cm"].to_numpy() for w in WINDOWS]))[0]

for window in WINDOWS:
    subset = window["steps"]
    print(f"{window['centre']:.0f} deg: {len(subset)} steps, {subset['event'].nunique()} events, "
          f"mean 2D step {subset['step_2d_cm'].mean():.2f} cm, "
          f"mean N {subset['spikes'].mean():.1f}, "
          f"N_eff {subset['inv_sqrt_n'].mean() ** -2:.1f}, "
          f"fit {window['intercept']:.2f} + {window['slope']:.2f}/sqrt(N)")

observed = (WINDOWS[1]["steps"]["step_2d_cm"].mean()
            - WINDOWS[0]["steps"]["step_2d_cm"].mean())
predicted = pooled_slope * (WINDOWS[1]["steps"]["inv_sqrt_n"].mean()
                            - WINDOWS[0]["steps"]["inv_sqrt_n"].mean())
print(f"observed difference   : {observed:+.2f} cm")
print(f"predicted from N alone: {predicted:+.2f} cm ({100 * predicted / observed:.0f} % of observed)")